# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR<sup>2</sup> dataset package (`Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells`) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is registered with a Croissant schema at:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Install mlcroissant if it's not already installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and explore the known record sets and fields using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata and print its overview
dataset = mlc.Dataset(croissant_url)

print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"DOI: {dataset.metadata.identifier}")
print(f"Version: {getattr(dataset.metadata, 'version', '')}")

## 2. Data Overview
List available Record Sets, their Fields, and reference their `@id` as required. This is crucial for accessing the right data via `mlcroissant`.

In [ ]:
# List all Record Sets and their Fields by @id for downstream reference
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):")

for i, rs in enumerate(record_sets, 1):
    print(f"\nRecord Set {i}:")
    print(f"  @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}")
        print(f"      Name: {field.name}")
        print(f"      DataType: {getattr(field, 'data_type', '')}")

## 3. Data Extraction
We will demonstrate how to load the main record set defined in the dataset. All references use `@id` values from the schema for clear and reproducible access.

In [ ]:
# Choose the main RecordSet (by @id) - usually the first record set is the data table
if len(record_sets) == 0:
    raise RuntimeError("No record sets found in this dataset. Please check the schema definition.")

main_record_set = record_sets[0]
main_record_set_id = main_record_set.id
print(f"Main record set @id: {main_record_set_id}")

# List available field @ids in the main record set
field_ids = [field.id for field in main_record_set.fields]
print("Fields in main record set:")
for f in main_record_set.fields:
    print(f"  - {f.id} (name={f.name}, dataType={getattr(f, 'data_type', None)})")

# Extraction: Load data into a pandas DataFrame using the main record set
records = list(dataset.records(record_set=main_record_set_id))
dataframe = pd.DataFrame(records)
print(f"\nLoaded {len(dataframe)} records with fields: {list(dataframe.columns)}")
dataframe.head()

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field (for example, protein abundance or coverage, as referenced by their `@id`) for common EDA steps: filtering, normalization, and grouping.

In [ ]:
# Selecting a numeric field for demonstration
# Replace with an actual numeric field @id from the printed overview - e.g., 'accession.coverage' or similar

# Try to automatically detect a numeric field from main_record_set.fields
numeric_field_id = None
for f in main_record_set.fields:
    if getattr(f, 'data_type', None) in ('Float', 'Integer', 'Number'):
        numeric_field_id = f.id
        print(f"Using numeric field: {numeric_field_id}")
        break
if numeric_field_id is None:
    # Arbitrarily choose the first field if cannot detect
    numeric_field_id = field_ids[0]
    print(f"No numeric field detected, using: {numeric_field_id}")

if numeric_field_id not in dataframe.columns:
    raise ValueError(f"Numeric field {numeric_field_id} not present in DataFrame columns.")

# Filtering: e.g. select values above the median
median_value = dataframe[numeric_field_id].median(skipna=True)
filtered_df = dataframe[dataframe[numeric_field_id] > median_value]
print(f"Filtered records where {numeric_field_id} > median ({median_value}): {len(filtered_df)} records")

# Normalization: zero mean, unit variance on the chosen field
filtered_df = filtered_df.copy()  # avoid pandas chained assignment warning
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optionally, group by a categorical field if available
group_field_id = None
for f in main_record_set.fields:
    if getattr(f, 'data_type', None) in (None, 'Text', 'String') and f.id != numeric_field_id:
        group_field_id = f.id
        break

if group_field_id and group_field_id in filtered_df.columns:
    print(f"\nGrouping by field: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(grouped_df.head())

## 5. Visualization
Visualizing the distribution of the numeric field and, if possible, its relationship to another field.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the main numeric field
plt.figure(figsize=(8, 4))
plt.hist(dataframe[numeric_field_id].dropna(), bins=30, alpha=0.7)
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.title(f'Distribution of {numeric_field_id}')
plt.show()

# Optionally, boxplot by group if one exists
if group_field_id and group_field_id in dataframe.columns:
    plt.figure(figsize=(10, 5))
    dataframe.boxplot(column=numeric_field_id, by=group_field_id, vert=False)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.suptitle('')
    plt.xlabel(numeric_field_id)
    plt.ylabel(group_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to:
- Load and inspect a FAIR<sup>2</sup>-compliant mass spectrometry dataset using the `mlcroissant` library
- Reference all data elements via their `@id` fields for transparent and reproducible data access
- Extract and analyze the main tabular record set, including typical EDA tasks and basic visualizations

You can now proceed with more detailed downstream analyses or modeling tasks using the loaded DataFrame. For full schema details or updates, see the [dataset metadata](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).